In [20]:
print("CSC 546 MinHash and LSH Project")
print("Jupyter Notebook is working!")

CSC 546 MinHash and LSH Project
Jupyter Notebook is working!


In [21]:
import pandas as pd
from pathlib import Path

# Locate the project folder.
cwd = Path.cwd()
project_root = cwd if (cwd / "data").exists() else cwd.parent

# Load the dataset from the data folder.
file_path = project_root / "data" / "All_Beauty.jsonl"
df = pd.read_json(file_path, lines=True, nrows=100000)

print("Dataset loaded successfully!")
print("Number of reviews:", len(df))

display(df.head())

Dataset loaded successfully!
Number of reviews: 100000


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,[],B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-05 14:08:48.923,0,True
1,4,Works great but smells a little weird.,"This product does what I need it to do, I just...",[],B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-04 18:10:55.070,1,True
2,5,Yes!,"Smells good, feels great!",[],B07PNNCSP9,B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,2020-05-16 21:41:06.052,2,True
3,1,Synthetic feeling,Felt synthetic,[],B09JS339BZ,B09JS339BZ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,2022-01-28 18:13:50.220,0,True
4,5,A+,Love it,[],B08BZ63GMJ,B08BZ63GMJ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,2020-12-30 10:02:43.534,0,True


In [22]:
# Data Cleaning
# Check how many reviews we loaded
print("Total reviews:", len(df))

# Check if any reviews are missing text
missing_reviews = df["text"].isna().sum()
print("Missing reviews:", missing_reviews)

# Count the number of words in each review
df["word_count"] = df["text"].fillna("").str.split().str.len()

# Look at the review length statistics
print("\nReview length statistics:")
print(df["word_count"].describe())

Total reviews: 100000
Missing reviews: 0

Review length statistics:
count    100000.000000
mean         38.424770
std          54.400559
min           0.000000
25%           9.000000
50%          22.000000
75%          46.000000
max        1531.000000
Name: word_count, dtype: float64


In [23]:
# Data Cleaning Pt2
#Goal: Clean the review text by removing punctuation, converting to lowercase, and removing extra spaces.
#Why: Cleaning is important because MinHash will compare groups of words. 
#We don't want punctuation or capitalization to make otherwise similar reviews appear different.
import re

# Function to clean the review text
def clean_text(text):

    # Check if the review is missing
    if not isinstance(text, str):
        return ""

    # Convert everything to lowercase
    text = text.lower()

    # Remove punctuation
    text = re.sub(r"[^\w\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# Apply the function to all reviews
df["clean_text"] = df["text"].apply(clean_text)

# Compare the original and cleaned reviews
display(df[["text", "clean_text"]].head(10))

,text,clean_text
0,This spray is really nice. It smells really go...,this spray is really nice it smells really goo...
1,"This product does what I need it to do, I just...",this product does what i need it to do i just ...
2,"Smells good, feels great!",smells good feels great
3,Felt synthetic,felt synthetic
4,Love it,love it
5,The polish was quiet thick and did not apply s...,the polish was quiet thick and did not apply s...
6,Great for many tasks. I purchased these for m...,great for many tasks i purchased these for mak...
7,These were lightweight and soft but much too s...,these were lightweight and soft but much too s...
8,This is perfect for my between salon visits. I...,this is perfect for my between salon visits i ...
9,I get Keratin treatments at the salon at least...,i get keratin treatments at the salon at least...


In [24]:
# Remove reviews that are too short
# Count words after cleaning
df["clean_word_count"] = df["clean_text"].str.split().str.len()

# Keep reviews with at least 2 words
df_clean = df[df["clean_word_count"] >= 2].copy()

# Check how many reviews were removed
print("Reviews before cleaning:", len(df))
print("Reviews after cleaning:", len(df_clean))
print("Reviews removed:", len(df) - len(df_clean))

# Display some cleaned reviews
display(df_clean[["clean_text", "clean_word_count"]].head(10))

# Test our cleaning function
test_review = "This PRODUCT is REALLY good!!!"

print("Original:", test_review)
print("Cleaned:", clean_text(test_review))

Reviews before cleaning: 100000
Reviews after cleaning: 97579
Reviews removed: 2421


,clean_text,clean_word_count
0,this spray is really nice it smells really goo...,62
1,this product does what i need it to do i just ...,47
2,smells good feels great,4
3,felt synthetic,2
4,love it,2
5,the polish was quiet thick and did not apply s...,24
6,great for many tasks i purchased these for mak...,22
7,these were lightweight and soft but much too s...,32
8,this is perfect for my between salon visits i ...,72
9,i get keratin treatments at the salon at least...,89


Original: This PRODUCT is REALLY good!!!
Cleaned: this product is really good


In [25]:
#Shingle Creation
# Create word shingles from our cleaned Amazon reviews.
# Shingles are groups of consecutive words that help us compare how similar
# two reviews are. We are starting with 2-word shingles because they keep
# some word order while still allowing us to find reviews with similar wording.
# These shingles will later be used for Jaccard similarity, MinHash, and LSH.


# Create word shingles from a review
def create_shingles(text, k=2):
    words = text.split()
    shingles = set()

    # Get each group of k consecutive words
    for i in range(len(words) - k + 1):
        shingle = tuple(words[i:i + k])
        shingles.add(shingle)

    return shingles


# Test with a sample review
review = "this product works really well"

print("2-word shingles:")
print(create_shingles(review, k=2))

print("\n3-word shingles:")
print(create_shingles(review, k=3))

# Create 2-word shingles for all cleaned reviews
df_clean["shingles"] = df_clean["clean_text"].apply(
    lambda text: create_shingles(text, k=2)
)

# Check the first review
print(df_clean["clean_text"].iloc[0])

print("\nShingles:")
print(df_clean["shingles"].iloc[0])

2-word shingles:
{('works', 'really'), ('product', 'works'), ('really', 'well'), ('this', 'product')}

3-word shingles:
{('works', 'really', 'well'), ('this', 'product', 'works'), ('product', 'works', 'really')}
this spray is really nice it smells really good goes on really fine and does the trick i will say it feels like you need a lot of it though to get the texture i want i have a lot of hair medium thickness i am comparing to other brands with yucky chemicals so i m gonna stick with this try it

Shingles:
{('it', 'smells'), ('so', 'i'), ('texture', 'i'), ('with', 'yucky'), ('i', 'm'), ('goes', 'on'), ('want', 'i'), ('does', 'the'), ('i', 'have'), ('i', 'am'), ('with', 'this'), ('other', 'brands'), ('to', 'other'), ('to', 'get'), ('will', 'say'), ('spray', 'is'), ('a', 'lot'), ('hair', 'medium'), ('though', 'to'), ('really', 'nice'), ('lot', 'of'), ('nice', 'it'), ('chemicals', 'so'), ('yucky', 'chemicals'), ('comparing', 'to'), ('like', 'you'), ('really', 'good'), ('is', 'really'),

In [26]:
import sys

# Let the notebook import code from the project's src folder.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.jaccard import jaccard_similarity

# Get the shingle sets for the first two cleaned reviews.
shingles_a = df_clean["shingles"].iloc[0]
shingles_b = df_clean["shingles"].iloc[1]

# Calculate their exact Jaccard similarity.
score = jaccard_similarity(shingles_a, shingles_b)

# Show the reviews and their similarity score.
print("Review A:", df_clean["clean_text"].iloc[0])
print("\nReview B:", df_clean["clean_text"].iloc[1])
print(f"\nJaccard similarity: {score:.4f}")
print("Meets the 0.80 cutoff:", score >= 0.80)

Review A: this spray is really nice it smells really good goes on really fine and does the trick i will say it feels like you need a lot of it though to get the texture i want i have a lot of hair medium thickness i am comparing to other brands with yucky chemicals so i m gonna stick with this try it

Review B: this product does what i need it to do i just wish it was odorless or had a soft coconut smell having my head smell like an orange coffee is offputting granted i did know the smell was described but i was hoping it would be light

Jaccard similarity: 0.0000
Meets the 0.80 cutoff: False


In [28]:
from itertools import combinations
from time import perf_counter

# Start with 1,000 reviews to keep the exact comparison manageable.
# Keep the original row indexes so we can identify matching reviews.
subset = df_clean.head(1000)

# Each entry contains a review's row index and its shingle set.
reviews = list(subset["shingles"].items())

# Store only pairs that meet our similarity threshold.
threshold = 0.80
matching_pairs = []
pairs_checked = 0

# Measure the time spent comparing pairs.
start = perf_counter()

# Generate each unique pair once.
# This avoids self-comparisons and reversed duplicates.
for (review_id_a, shingles_a), (review_id_b, shingles_b) in combinations(
    reviews, 2
):
    score = jaccard_similarity(shingles_a, shingles_b)
    pairs_checked += 1

    # Include pairs scoring exactly 0.80 as well as higher scores.
    if score >= threshold:
        matching_pairs.append({
            "review_id_1": review_id_a,
            "review_id_2": review_id_b,
            "similarity": score
        })

elapsed = perf_counter() - start

# Specify columns so the table works even if no matches are found.
results = pd.DataFrame(
    matching_pairs,
    columns=["review_id_1", "review_id_2", "similarity"]
)

print("Reviews compared:", len(subset))
print("Pairs checked:", pairs_checked)
print("Pairs with similarity >= 0.80:", len(results))
print(f"Comparison time: {elapsed:.2f} seconds")

display(results.head(10))

Reviews compared: 1000
Pairs checked: 499500
Pairs with similarity >= 0.80: 38
Comparison time: 1.65 seconds


,review_id_1,review_id_2,similarity
0,4,415,1.000000
1,4,619,1.000000
2,38,116,1.000000
3,163,485,1.000000
4,241,243,0.803468
5,254,256,0.813253
6,374,375,1.000000
7,415,619,1.000000
8,417,659,1.000000
9,539,937,1.000000


In [29]:
# Create a folder for experiment results.
output_dir = project_root / "results"
output_dir.mkdir(parents=True, exist_ok=True)

# Save all qualifying pairs and their exact similarity scores.
results.to_csv(output_dir / "jaccard_pairs_1000.csv", index=False)

# Save which reviews were compared so MinHash and LSH
# can be evaluated on the exact same subset.
subset[["clean_text"]].to_csv(
    output_dir / "jaccard_subset_1000.csv",
    index=True,
    index_label="review_id"
)

print("Results and review subset saved to:", output_dir)

Results and review subset saved to: /Users/roquibatadetunji/csc 546/CSC-546-Project/results
